# Phase II: Data Collection and Exploratory Data Analysis
##### INFO2950 Project by: Anderson Ramirez (ar2527) and Jackson Rhamy (jtr84)
##### GitHub Repository: [View Project Repository](https://github.com/RamirezDevOps/info2950_data_project)

#### Package Imports

We will import Python libraries to manipulate and visualize our data.

In [1]:
import pandas as pd
import numpy as np
import duckdb

### Data Collection and Cleaning

#### President Biden's Approval Ratings

The Where and What of this dataset:
- **Gallup Presidential Approval Ratings: Joe Biden**, provided by Gallup.
- The dataset can be downloaded as a CSV from the following link: https://news.gallup.com/poll/329384/presidential-approval-ratings-joe-biden.aspx

Why this dataset:
- This dataset provides information about Joe Biden’s presidential approval rating over time, allowing us to examine how his approval fluctuated during his presidency and **compare those changes with the frequency of his mentions in political podcasts**.

##### Loading President Biden's Approval Ratings CSV Data

Here we are loading the data, printing its shape and DataFrame.

In [2]:
# Load the CSV file containing Biden's approval ratings into a Pandas DataFrame
biden_ratings_df = pd.read_csv("../data/raw/bidenApprovalRating.csv")

print(f"The shape of biden_ratings_df is {biden_ratings_df.shape}")

biden_ratings_df.head()

##### Remove Headers, Unnecessary Columns, Rename Columns, and Convert Values

From the printed DataFrame, we can see that some rows have NaN values. These rows serve as headers that organize the data. For example, the first row specifies the units, in this case percentages, for the corresponding columns, while `<b>2025</b>` is a heading for the data collected in 2025.

Since these rows only provide additional information about the data rather than actual observations, we will **remove the rows with NaNs**. We will also **remove the `Disapprove` and `No opinion` columns** because our research questions focus on the approval rating specifically. 

Next, we will rename the columns to make their labels more descriptive, and **convert the percentage columns from an object to numeric data types** to make sure they can be used for analysis. Since these values represent percentages, **we will divide them by 100 to express them as a proportion** between 0 and 1. This way the dataset will only have relevant observations with clear variable names and appropriate data types.

In [3]:
# Remove any rows with missing data and reset the row numbers to start from 0
biden_ratings_df = biden_ratings_df.dropna().reset_index(drop=True)

# Drop the columns we don't need
biden_ratings_df = biden_ratings_df.drop(columns=['Disapprove', 'No opinion'])

# Rename the columns to clearer names
biden_ratings_df = biden_ratings_df.rename(columns={'X.1':'Date', 'Approve':'Approval_Proportion'})

# Ensure the approval column is treated as decimal numbers
biden_ratings_df = biden_ratings_df.astype({'Approval_Proportion':float})

# Convert the approval percentages into true proportions
biden_ratings_df['Approval_Proportion'] = biden_ratings_df['Approval_Proportion'] / 100

biden_ratings_df.head()

In [4]:
# Creating intermediate saving point.
biden_ratings_df.to_csv('../data/clean/bidenApprovalRatingClean.csv', index=False)

##### Format the Date column

From the original printed DataFrame, we see that the dates are not consistently formatted, some months have duplicate observations, and two polling periods span multiple months. For example, October 2024 has two observations of .41 and .39, while all dates have polling ranges, like 2024 Oct 14-27, rather than standard monthly dates like 2024-10. Additionally, two polling periods cross month boundaries, 2022 Nov 9-Dec 2 and 2021 Jan 21-Feb 2.

- The first date issue we will tackle is the polling periods that span two months. To handle these cases, we will **assign each polling observation to the month containing the midpoint of its polling period**. This is consistent with the methodology explained by political scientists Will Jennings and Christopher Wlezien in a 2013 paper, where they use the midpoint between a survey's fieldwork start and end dates to find its associated date (see References section for more information). Rather than arbitrarily assigning a survey to its start or end date, this method uses the midpoint of the actual data collection period.
    - While assigning polling observations to their corresponding months using the midpoint of each polling period,   we found that December 2022 had no approval observation. To keep a consistent monthly time series, we **added a row for December 2022 with a missing approval proportion (`NaN`)**. We chose to **preserve this missing value rather than estimate or interpolate it** because doing so would introduce an approval proportion that was not directly observed in our original dataset. This approach follows the time-series data cleaning practices discussed in Lecture 7, which emphasize maintaining consistent time intervals, explicitly representing missing observations, and documenting how missing data is handled.

As a result, our cleaned dataset contains **47 monthly observations, of which 46 have recorded approval proportions**. Analyses requiring approval proportions will use the 46 nonmissing observations.
- Next, we will address the presence of multiple observations within the same month. To resolve this, we will **convert the Date column object type to a Datetime type**, and then find the **average approval rating for each month**, that has multiple observations, using temporal aggregation, which combines higher frequency observations into a lower frequency, monthly dataset. **Each poll is weighted equally** regardless of its sample size or methodology; see our limitations section for more information. We chose this approach because our goal is to obtain one approval value per month for comparison with our podcast data, rather than develop a more sophisticated polling model.

- Finally, we are **removing January 2021 and January 2025** from our data because Joe Biden was not president for the full month in either case. Since he took office on January 20, 2021, and left on January 20, 2025, these two months only offer a few days of data instead of a complete 30 or 31 days. Because our research compares monthly presidential approval ratings against monthly podcast mentions, every data point needs to look at an identical timeframe. Keeping these partial months could skew the results and make the comparison unfair. Removing them leaves us with **47 complete, consistent months running from February 2021 through December 2024**.

In [5]:
# Loading intermediate data
biden_ratings_df = pd.read_csv('../data/clean/bidenApprovalRatingClean.csv')

# Replace specific cross-month date ranges with representative single dates for consistency
biden_ratings_df['Date'] = biden_ratings_df['Date'].replace({'2021 Jan 21-Feb 2':'2021 Jan 27', '2022 Nov 9-Dec 2':'2022 Nov 20'})

# Add a missing December 2022 observation with a NaN approval proportion
biden_ratings_df = pd.concat([biden_ratings_df, pd.DataFrame({'Date': ['2022 Dec 20'], 'Approval_Proportion':[np.nan]})], ignore_index=True)

# Convert dates to datetime format and standardize them to the first day of each month
biden_ratings_df['Date'] = pd.to_datetime(biden_ratings_df['Date'].str.split('-').str[0]).dt.to_period('M').dt.to_timestamp()

# Group observations by month and calculate the average approval proportion
# Sort the results chronologically by date
biden_ratings_df = duckdb.sql('SELECT Date, AVG("Approval_Proportion") AS "Approval_Proportion", \
                                            FROM biden_ratings_df GROUP BY Date ORDER BY Date').df()

# Exclude partial months
biden_ratings_df = biden_ratings_df[biden_ratings_df['Date'].between('2021-02-01', '2024-12-01')].reset_index(drop=True)

biden_ratings_df.head()

In [6]:
# Saving biden_ratings_df to a csv
biden_ratings_df.to_csv('../data/clean/bidenApprovalRatingClean.csv', index=False)

#### Political Podcast Data

The Where and What of this dataset:
- The dataset comes from the **Brookings Institution’s Popular Political Podcast Dataset**, which provides downloadable CSV data.
- It contains metadata for episodes of popular political podcasts, including **show names, publication dates, titles, descriptions, and assigned political leanings**.

Why this dataset:
- We use this dataset to **measure how frequently Joe Biden is mentioned** in political podcast episode metadata over time and **examine how mention rates vary by political leaning**, allowing us to compare these patterns with his presidential approval ratings.

##### Loading Political Podcast Data

Before cleaning our podcast dataset, we need to examine its structure to understand what information is available and find any potential issues that could affect our analysis. We will start by inspecting the dimensions, columns names, data types, and a sample of observations. This way we can see which columns are relevant and what cleaning steps are needed.

In [7]:
# Load the full podcast dataset from a CSV
podcasts_df = pd.read_csv("../data/raw/podcast_full-dataset.csv")

print(f"The shape of podcasts_df is {podcasts_df.shape}")

# Display column names, data types, and non-null counts
print(podcasts_df.info())

podcasts_df.head()

##### Removing Unnecessary Columns and Handling Missing values

From the initial inspection, we found that multiple columns are not necessary for answering our research questions. Since our analysis focuses on the frequency of Biden mentions in podcast metadata, we will **only keep `Show Name`, `Full Date`, `Title`, `Description`, and `Partisan Leaning`**. The remaining columns will be dropped because they do not contribute to our analysis.

We also saw that the **`Description` column is missing 24 values**. Instead of removing these 24 episodes, we will **replace these values with empty strings**. This way we can **keep the observations and still find mentions of Biden via the episode titles**, even if they have a missing description.

In [8]:
# Remove unnecessary columns from the podcast dataset
podcasts_df = podcasts_df.drop(columns=['Episode ID', 'Unnamed: 0', 'Category on Apple', 'Entered Dataset', 'Audio URL'])

# Replace missing descriptions with empty strings
podcasts_df['Description'] = podcasts_df['Description'].fillna('')

print(f"The shape of podcasts_df is {podcasts_df.shape}")

# Check for remaining missing values in each column
print(f"Missing values after cleaning: \n{podcasts_df.isna().sum()}")

# Creating intermediate saving point.
podcasts_df.to_csv('../data/clean/podcastDataCleaned.csv', index=False)

podcasts_df.head()

##### Find Duplicates

We want to check if the dataset has any duplicate episodes. **Duplicate observations can affect our results** by causing certain episodes to be counted more than once, potentially inflating Biden's mentions.

We will check for **completely identical rows**, and then if any episodes share the **same `Show Name`, `Full Date`, and `Title`**, as this combination might show duplicate episodes.

In [9]:
print(f"Duplicate rows: {podcasts_df.duplicated().sum()}")

print(f"Potential duplicate episodes: {podcasts_df.duplicated(subset=['Show Name', 'Full Date', 'Title']).sum()}")

The check found **144 completely identical rows** and **192 potential duplicated episodes** based on matching `Show Name`, `Full Date`, and `Title`. It is very possible that some episodes appear multiple times in the dataset. However, since the number of completely identical rows is lower, we will **investigate these observations more before removing them**.

In [10]:
# Find duplicate podcast episodes based on show name, date, and title
# Keep all occurrences of duplicates for inspection
duplicate_episodes = podcasts_df[podcasts_df.duplicated(subset=['Show Name', 'Full Date', 'Title'], keep=False)]

# Sort duplicate episodes by show name, date, and title
duplicate_episodes = duplicate_episodes.sort_values(by=['Show Name', 'Full Date', 'Title'])

duplicate_episodes.head(10)

##### Removing Duplicate Episodes

After seeing the potentially duplicated observations, we found that some episodes are **completely identical across all five columns**, while others share the **same show, publication date, and the title but had different descriptions**.

Since descriptions are important to finding Biden mentions, we decided to **only remove the 144 identical rows**. We **keep the 48 episodes with different descriptions** because they might represent different content, even if it shares the same title and date.

In [11]:
# Remove duplicate rows and reset the DataFrame index
podcasts_df = podcasts_df.drop_duplicates().reset_index(drop=True)

# Check the number of remaining duplicate rows
print(f"Remaining duplicate rows: {podcasts_df.duplicated().sum()}")

print(f"The shape of podcasts_df now is {podcasts_df.shape}")

##### Filtering Episodes to Biden's Presidency 

Because our research focuses on podcast episodes published during Biden's presidency, we will filter `podcasts_df` to include only episodes published from **February 1, 2021, through December 31, 2024**. This keeps our podcast data **aligned with the 47 complete months** isolated for our analysis.

In the initial inspection, we saw that `Full Date` is stored as an object rather than a datetime data type. Thus, we will **convert this column into a Pandas datetime object** to conduct some chronological filtering.

We will then filter `podcasts_df` directly to **keep only episodes published during Biden's presidency**, rather than making a separate DataFrame. Episodes outside this time frame will be excluded because they are not relevant to our research questions. We will also reset the index of the filtered DataFrame so that observations are numbered consecutively. 

Finally, we will compare the dimensions of the dataset before and after filtering and make sure the publication dates match our desired time frame.

In [12]:
# Convert the 'Full Date' column to datetime format
podcasts_df['Full Date'] = pd.to_datetime(podcasts_df['Full Date'])

# Store the original dataset shape before filtering
og_shape = podcasts_df.shape

# Filter episodes to include only those published from February 01, 2021, through December 31, 2024
podcasts_df = podcasts_df[podcasts_df['Full Date'].between('2021-02-01', '2024-12-31')]

# Reset the DataFrame index after filtering
podcasts_df = podcasts_df.reset_index(drop=True)

# Display the original and filtered dataset dimensions
print(f"Original dataset shape: {og_shape}")
print(f"Filtered dataset shape: {podcasts_df.shape}")

# Check the earliest and latest publication dates in the filtered dataset
print(f"Earliest date: {podcasts_df['Full Date'].min()}")
print(f"Latest date: {podcasts_df['Full Date'].max()}")

# Creating intermediate saving point.
podcasts_df.to_csv('../data/clean/podcastDataCleaned.csv', index=False)

podcasts_df.head()

##### Find Biden Mentions in Podcast Metadata

To find which podcast episodes mention Biden, we will parse through the Title and Description columns using Pandas `str.contains()` method. 

First, we will check each column separately for the word "Biden," with **case-insensitive matching to account for differences in capitalization and a regular expression to match the entire word**. Because Hunter Biden was a massive presence in the general news cycle due to this criminal trial and legal troubles, his media coverage could skew our data. To make sure this does not happen, we will explicitly **exclude any mentions of Hunter Biden** in our count.

Finally, we will combine the remaining results from both columns to create a binary variable called `Biden_mention`, where **1 means Biden is in the title or description and 0 indicates he is not**. **Episodes where Biden is mentioned in both columns will only be counted once**.

In [13]:
# Find all mentions of "Biden"
title_biden = podcasts_df['Title'].str.contains(r"\bBiden\b", case=False, na=False)
description_biden = podcasts_df['Description'].str.contains(r"\bBiden\b", case=False, na=False)

# Combine them 
podcasts_df['Biden_mention'] = (title_biden + description_biden).astype(int)

# Find rows that mention "Hunter Biden"
has_hunter_title = podcasts_df['Title'].str.contains("Hunter Biden", case=False, na=False)
has_hunter_desc = podcasts_df['Description'].str.contains("Hunter Biden", case=False, na=False)

# If "Hunter Biden" is in the title OR description, set 'Biden_mention' back to 0
podcasts_df.loc[has_hunter_title | has_hunter_desc, 'Biden_mention'] = 0

print(podcasts_df["Biden_mention"].value_counts())

podcasts_df[["Title", "Description", "Biden_mention"]].head()

##### Grouping Episodes by Month

To find the relationship between Biden's monthly approval rating and his mention frequency in political podcasts, we will organize the dataset by month.

We will create a `Month` column using the `Full Date` column to identify the year and month each episode was published. Then, we will **group the episodes by month** and calculate the **total number of episodes published and the number of episodes mentioning Biden**. These values will help us calculate the **monthly Biden mention rate as the proportion of episodes mentioning Biden out of all episodes published that month**. These results will be stored in a separate DataFrame called `monthly_podcasts_df` while preserving the original episode data.

In [14]:
# Extract the year and month from each episode's publication date
podcasts_df["Month"] = podcasts_df["Full Date"].dt.to_period("M")

# Group episodes by month and calculate the total number of episodes and Biden mentions
monthly_podcasts_df = podcasts_df.groupby("Month").agg(
    Total_episodes=("Biden_mention", "count"),
    Biden_episodes=("Biden_mention", "sum")
).reset_index()

# Calculate the monthly proportion of episodes mentioning Biden
monthly_podcasts_df["Mention_Proportion"] = (monthly_podcasts_df["Biden_episodes"] / monthly_podcasts_df["Total_episodes"])

# Creating intermediate saving point.
podcasts_df.to_csv('../data/clean/podcastDataCleaned.csv', index=False)
monthly_podcasts_df.to_csv('../data/clean/monthlyPodcastData.csv', index=False)

monthly_podcasts_df.head()

##### Inspect Political Leaning Column

We need to **verify that our Partisan Leaning column is consistent** because one of our research questions examines whether the relationship between Biden's approval rating and podcast mention frequency differs by political leaning.

We will see unique values in this column and **count the number of observations belonging to each classification**.

In [15]:
print(podcasts_df["Partisan Leaning"].unique())
print(podcasts_df["Partisan Leaning"].value_counts())

print(f"Missing values: {podcasts_df["Partisan Leaning"].isna().sum()}")

The `Partisan Leaning` column has **four categories, `More Liberal`, `More Conservative`, `Moderate`, and `Unknown`**. There are **no missing values**, so no additional cleaning is necessary. We will **keep the `Unknown` category to preserve all observations, but exclude it when comparing political leanings**.

### Combined Monthly Dataset

The Where and What of this dataset:
- This is a derived dataset created by **merging Gallup’s Biden presidential approval ratings with monthly statistics from the Brookings political podcast dataset**, producing **47 monthly observations and nine columns covering February 2021 through December 2024**.

Why this dataset:
- We created this dataset to examine the **relationship between Biden’s monthly presidential approval ratings and the proportion of political podcast episodes mentioning him**, including **how this relationship varies across liberal, moderate, and conservative podcasts**.

##### Prepare the Datasets for Merging

Before merging the two datasets, we need to make sure the month columns use the same format and that their measurements are consistent.

In `biden_ratings_df`, the `Date` column represents the first day of each month, `Approval_Proportion` is already expressed as a proportion between 0 and 1. In `monthly_podcasts_df`, `Month` is stored as a monthly period, while `Mention_Proportion` is expressed as a decimal, ranging from 0 to 1.

We will make copies of both datasets to keep our previous cleaning work. Then, we will **rename and convert the date columns into a common `Month` format**. We will also **recalculate the podcast mention rate as a proportion** using the number of episodes that mention Biden divided by the total number of episodes.

Finally, we will conduct a **sanity check for duplicate months in either dataset**, since each month should only have one observation before we merge the data.

In [16]:
# Load Datasets
approval_monthly = pd.read_csv('../data/clean/bidenApprovalRatingClean.csv')
podcast_monthly = pd.read_csv('../data/clean/monthlyPodcastData.csv')

# Renames the approval date column and standardizes both month columns to the first day of each month.
approval_monthly = approval_monthly.rename(columns={"Date": "Month"})

approval_monthly['Month'] = pd.to_datetime(approval_monthly['Month']).dt.to_period('M').dt.to_timestamp()

podcast_monthly['Month'] = pd.to_datetime(podcast_monthly['Month'].astype(str)).dt.to_period('M').dt.to_timestamp()

# Recalculate the podcast mention proportion from counts
podcast_monthly["Mention_Proportion"] = podcast_monthly["Biden_episodes"] / podcast_monthly["Total_episodes"]

# Keep only columns needed
approval_monthly = approval_monthly[["Month", "Approval_Proportion"]]
podcast_monthly = podcast_monthly[["Month", "Total_episodes", "Biden_episodes", "Mention_Proportion"]]

# Verify there is one row per month
print(f"Duplicate approval months: {approval_monthly["Month"].duplicated().sum()}")
print(f"Duplicate podcast months: {podcast_monthly["Month"].duplicated().sum()}")

print(approval_monthly.head())
print(podcast_monthly.head())

##### Merging the Datasets

Now that both datasets have been standardized, we can combine them using the Month column. We will use a **left join with the podcast dataset as the primary table** to **keep all months with podcast observations, even if the corresponding approval proportions are missing**.

The combined dataset will contain the monthly approval proportion, total podcast episodes, episodes mentioning Biden, and the proportion of episodes mentioning Biden.

To address our subquestion about political leaning, we will also use our cleaned episode-level podcast data to **calculate monthly Biden mention proportions for conservative, moderate, and liberal podcasts**. These proportions will be added as separate columns to our combined dataset, allowing us to compare political leanings while **keeping one observation per month**.

Additionally, we will create a new variable that measures the monthly difference in Biden mention proportions between liberal and conservative podcasts. We will calculate this by **subtracting the conservative mention proportion from the liberal mention proportion (Liberal_Proportion - Conservative_Proportion)**. A **positive value indicates that liberal podcasts mentioned Biden more frequently, while a negative value indicates that conservative podcasts mentioned him more frequently**. This will allow us to examine how the difference in mention proportions between the two political leanings changes over time and relates to Biden's approval proportion.

After merging, we will **check for missing values** to identify months without approval ratings or podcast information. We will also **verify that the previously identified missing approval proportion for December 2022 remains** in the combined dataset.

In [17]:
# Merge monthly podcast data with approval ratings.
combined_monthly_df = duckdb.sql('SELECT p.Month, p.Total_episodes, p.Biden_episodes, p.Mention_Proportion, \
                                        a.Approval_Proportion FROM podcast_monthly AS p LEFT JOIN \
                                        approval_monthly AS a ON p.Month = a.Month ORDER BY p.Month').df()

# Loading the cleaned episode level podcast dataset
podcasts_leaning = pd.read_csv("../data/clean/podcastDataCleaned.csv")

# Standardize the Month column
podcasts_leaning["Month"] = pd.to_datetime(podcasts_leaning["Month"]).dt.to_period("M").dt.to_timestamp()

# Group episodes by month and political leaning
leaning_monthly = podcasts_leaning.groupby(["Month", "Partisan Leaning"]).agg(
    Total_episodes=("Biden_mention", "count"),
    Biden_episodes=("Biden_mention", "sum")
).reset_index()

# Calculate mention proportions for each political leaning
leaning_monthly["Mention_Proportion"] = leaning_monthly["Biden_episodes"] / leaning_monthly["Total_episodes"]

# Turn political leaning categories into separate columns
leaning_wide = leaning_monthly.pivot(index="Month", columns="Partisan Leaning", values="Mention_Proportion").reset_index()

# rename the political leaning columns
leaning_wide = leaning_wide.rename(columns={
    "More Conservative": "Conservative_Mention",
    "Moderate": "Moderate_Mention",
    "More Liberal": "Liberal_Mention",
    "Unknown": "Unknown_Mention"
})

# Add political leaning mention proportions to the combined data
combined_monthly_df = combined_monthly_df.merge(leaning_wide, on='Month', how='left')

# Calculate the monthly difference between liberal and conservative mentions
combined_monthly_df["Liberal_Conservative_Difference"] = combined_monthly_df["Liberal_Mention"] - combined_monthly_df["Conservative_Mention"]

# Check the shape of merged dataset
print(f"Combined dataset shape: {combined_monthly_df.shape}")

# Check for missing values and display them
print(combined_monthly_df.isna().sum())
print(combined_monthly_df[combined_monthly_df.isna().any(axis=1)])

combined_monthly_df.head()

In [18]:
# Creating intermediate saving point.
combined_monthly_df.to_csv('../data/clean/combinedMonthlyProportions.csv', index=False)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=ffbb765c-73bd-48fb-b032-b98a4e6049ec' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>